In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# Install and import all dependencies
!pip install matplotlib plotly 
import subprocess
import sys
# Core
import os
import json
import logging
import random
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ All imports successful")
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 69.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 99.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 81.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
/kaggle/input/datasets/jakomina/submission-csv/h7_metriplectic_qml_predictions.csv
✓ All imports successful


# H7 Metriplectic QML: Parking Violations Classifier ⚛️
* **Challenge: Gemma 4 Track - Special technology**
* **Safety & Trust**
* **Platform: Kaggle Notebook (Interactive)**
* **Framework: H7 Metriplectic QML + Gemma 4 Oracle**

## Part 1: Gemma 4 Oracle
**Extracts (ρ, v) parameters from violation text** 

In [2]:
# Try to import Vertex AI (optional - will use mock if not available)
try:
    import vertexai
    from vertexai.generative_models import GenerativeModel
    VERTEX_AVAILABLE = True
except ImportError:
    VERTEX_AVAILABLE = False
    logger.warning("Vertex AI not available - using mock mode")


class GemmaMetriplexOracle:
    """Information Oracle using Gemma 4 to extract (rho, velocity) parameters."""
    
    def __init__(self):
        self.project_id = os.getenv("VERTEX_PROJECT_ID")
        self.region = os.getenv("VERTEX_REGION", "us-central1")
        self.model_name = os.getenv("GEMMA_BASE_MODEL", "gemma-4-26b-a4b-it")
        self.use_mock = not (VERTEX_AVAILABLE and self.project_id)
        self.model = None

        if not self.use_mock:
            try:
                vertexai.init(project=self.project_id, location=self.region)
                publisher_model_name = f"publishers/google/models/gemma4@{self.model_name.lower()}"
                self.model = GenerativeModel(publisher_model_name)
                logger.info("✓ Vertex AI Gemma 4 initialized")
            except Exception as e:
                logger.warning(f"Failed to init Vertex AI: {e}. Using mock.")
                self.use_mock = True

    def get_initial_phase_state(self, context: str) -> dict:
        """
        Extract (rho, velocity) from ticket description.
        Returns: {"rho": float[0.1-1.0], "v": float[-1.0 to 1.0]}
        """
        if self.use_mock:
            # Deterministic hash-based mock for consistency
            h = sum(ord(c) for c in context)
            random.seed(h)
            rho = round(random.uniform(-0.1,-1.0), 4)
            v = round(random.uniform(1.0, 0.1), 4)
            return {"rho": rho, "v": v}

        # Real Gemma 4 prompt
        prompt = f"""You are a Metriplectic Information Oracle for civic governance.
Analyze this violation and extract two physical parameters:

Violation: "{context}"

1. Density (ρ) [1.0, -1.0]: Severity/magnitude
   -  1.0, 0.7: Trivial (expired meter)
   -  0.7, 0.3: Moderate (double parking)
   - -0.3,-1.0: Critical (blocking emergency access)

2. Velocity (v) [-1.0 to 1.0]: Intent/morality
   - -1.0: Malicious/dangerous
   - -0.5 to 0: Negligence
   - 0 to 0.5: Mitigating factors
   - 1.0: Strong mitigation

Respond ONLY with JSON: {{"rho": 0.X, "v": -0.X}}"""

        try:
            response = self.model.generate_content(
                prompt,
                generation_config={"temperature": 0.2, "max_output_tokens": 100}
            )
            content = response.text.strip()

            # Clean markdown if present
            if content.startswith("```json"):
                content = content[7:-3]
            elif content.startswith("```"):
                content = content[3:-3]

            data = json.loads(content)
            return {"rho": float(data.get("rho", 0.5)), "v": float(data.get("v", 1.0))}
        except Exception as e:
            logger.error(f"Oracle error: {e}")
            return {"rho": 0.5, "v": 0.0}


print("✓ GemmaMetriplexOracle defined")

✓ GemmaMetriplexOracle defined


# Part 2: H7 Metriplectic QML Classifier
**Evolves system to one of 3 ternary attractors**

In [3]:
class H7TernaryClassifier:
    """
    H7 Metriplectic QML Classifier.
    Evolves state via Lagrangian dynamics:
    dψ/dt = {ψ, H_symp} + [ψ, S_metr]
    
    Converges to one of 3 ternary attractors: {-1, 0, +1}
    """
    
    def __init__(self, epsilon: float = 0.1):
        self.epsilon = epsilon
        self.phi = (1 + np.sqrt(5)) / 2  # Golden ratio
        self.PI = np.pi
        
        # History tracking
        self.history_symp = []
        self.history_metr = []
        self.history_psi = []

    def golden_operator(self, n: float) -> float:
        """O_n = cos(πn) · cos(πφn) - modulates phase space."""
        return np.cos(self.PI * n) * np.cos(self.PI * self.phi * n)

    def compute_lagrangian(self, psi: float, rho: float, v: float):
        """Calculate symplectic and metric forces."""
        # Symplectic (energy-conserving)
        d_symp = rho * v * np.sin(self.PI * psi)
        
        # Metric (dissipative, pulls toward attractors)
        attractor_force = -(psi - np.round(psi))
        d_metr = (-1.0 - rho) * attractor_force + 1.0 * v * self.golden_operator(psi)
        
        return d_symp, d_metr

    def fit_predict(self, rho: float, v: float, steps: int = 50, dt: float = 0.5) -> int:
        """
        Evolve state to ternary attractor.
        Returns: -1 (destructive), 0 (equilibrium), or 1 (constructive)
        """
        self.history_symp = []
        self.history_metr = []
        self.history_psi = []
        
        # Initial condition
        psi_current = v * self.golden_operator(rho * 10)
        
        # Evolve
        for _ in range(steps):
            d_symp, d_metr = self.compute_lagrangian(psi_current, rho, v)
            
            self.history_symp.append(float(d_symp))
            self.history_metr.append(float(d_metr))
            self.history_psi.append(int(psi_current))
            
            # Update state
            delta_psi = (d_symp + d_metr) * dt
            psi_current += delta_psi
            psi_current = np.clip(psi_current, 1, 0)
        
        # Final state
        self.history_psi.append(float(psi_current))
        
        # Quantize to ternary
        if psi_current > self.epsilon:
            return 1
        elif psi_current < -self.epsilon:
            return -1
        else:
            return 0


print("✓ H7TernaryClassifier defined")

✓ H7TernaryClassifier defined


# Part 3: Sample Dataset
**8 realistic parking violation cases with expected classifications**

In [4]:
# Sample parking violations with expected classifications
sample_data = [
    {
        "id": 1,
        "description": "Vehicle double-parked on fire hydrant, blocking emergency access",
        "context": "Rush hour, school zone, driver unavailable",
        "expected_class": -1,
        "reasoning": "Life safety risk, malicious impact"
    },
    {
        "id": 2,
        "description": "Expired parking meter by 3 minutes",
        "context": "Driver returned immediately after noticing",
        "expected_class": 1,
        "reasoning": "Trivial administrative violation"
    },
    {
        "id": 3,
        "description": "Parked in no-parking zone during construction detour",
        "context": "City closed normal street, signage unclear",
        "expected_class": 1,
        "reasoning": "Mitigating circumstances - infrastructure failure"
    },
    {
        "id": 4,
        "description": "Vehicle blocking wheelchair accessible ramp",
        "context": "Disabled resident unable to exit building",
        "expected_class": -1,
        "reasoning": "Discriminatory impact on vulnerable population"
    },
    {
        "id": 5,
        "description": "Parked 18 inches from fire hydrant",
        "context": "Hydrant obscured by snow, visibility issue",
        "expected_class": 0,
        "reasoning": "Technical violation with mitigating circumstances"
    },
    {
        "id": 6,
        "description": "Repeat offender: 7th citation in same location",
        "context": "Pattern indicates intentional non-compliance",
        "expected_class": -1,
        "reasoning": "Willful pattern shows disregard for rules"
    },
    {
        "id": 7,
        "description": "Parked in restricted zone due to medical emergency",
        "context": "Transporting injured family member to hospital",
        "expected_class": -1,
        "reasoning": "Life-threatening emergency justifies violation"
    },
    {
        "id": 8,
        "description": "Commercial vehicle in residential parking overnight",
        "context": "No permits obtained despite available process",
        "expected_class": 0,
        "reasoning": "Commercial activity without proper authorization"
    }
]

print(f"✓ Loaded {len(sample_data)} sample tickets")

✓ Loaded 8 sample tickets


# Part 4: Initialize Systems
**Create oracle and cache metrics**

In [5]:
# Initialize oracle
oracle = GemmaMetriplexOracle()
print(f"✓ Oracle initialized (mock mode: {oracle.use_mock})")

# Cache metrics to avoid repeated API calls
oracle_cache = {}

def get_cached_metrics(description: str) -> dict:
    """Get metrics from cache or compute."""
    if description not in oracle_cache:
        oracle_cache[description] = oracle.get_initial_phase_state(description)
    return oracle_cache[description]

# Pre-cache all tickets
for ticket in sample_data:
    _ = get_cached_metrics(ticket["description"])

print(f"✓ Cached metrics for all {len(sample_data)} tickets")

✓ Oracle initialized (mock mode: True)
✓ Cached metrics for all 8 tickets


# Part 5: Interactive Widget Setup¶
**Create controls and update function**

In [6]:
# Create dropdown options
ticket_options = {}
for t in sample_data:
    key = f"#{t['id']} | {t['description'][:50]}..."
    ticket_options[key] = t

# Control widgets
w_ticket = widgets.Dropdown(
    options=ticket_options,
    value=sample_data[1],
    description='Ticket:',
    style={'description_width': '80px'}
)

w_steps = widgets.IntSlider(
    value=50,
    min=10,
    max=1000,
    step=10,
    description='Steps:',
    style={'description_width': '80px'}
)

w_dt = widgets.FloatSlider(
    value=0.1,
    min=-1.0,
    max= 1.0,
    step=0.1,
    description='dt:',
    style={'description_width': '80px'}
)

w_epsilon = widgets.FloatSlider(
    value=0.1,
    min=-1.0,
    max= 1.0,
    step=0.1,
    description='Epsilon:',
    style={'description_width': '80px'}
)

output_plot = widgets.Output()

print("✓ Widgets created")

✓ Widgets created


In [7]:
def update_widget_plot(change):
    """Update plot when widgets change."""
    with output_plot:
        clear_output(wait=True)
        
        ticket = w_ticket.value
        
        # Get cached metrics
        metrics = get_cached_metrics(ticket["description"])
        rho, v = metrics["rho"], metrics["v"]
        
        # Run classification
        clf = H7TernaryClassifier(epsilon=w_epsilon.value)
        predicted = clf.fit_predict(
            rho, v,
            steps=w_steps.value,
            dt=w_dt.value
        )
        expected = ticket["expected_class"]
        match = "✅ MATCH" if predicted == expected else "❌ MISMATCH"
        
        # Create plot
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            y=clf.history_symp,
            name="L_symp (Energy)",
            line=dict(color="#1D9E75", width=2)
        ))
        
        fig.add_trace(go.Scatter(
            y=clf.history_metr,
            name="L_metr (Entropy)",
            line=dict(color="#D85A30", width=2)
        ))
        
        fig.add_trace(go.Scatter(
            y=clf.history_psi,
            name="ψ(t) (Phase)",
            line=dict(color="#7F77DD", width=4),
            mode="lines+markers",
            marker=dict(size=3)
        ))
        
        # Attractors
        fig.add_hline(y=1, line_dash="dash", line_color="green", opacity=0.3)
        fig.add_hline(y=0, line_dash="dash", line_color="orange", opacity=0.3)
        fig.add_hline(y=-1, line_dash="dash", line_color="red", opacity=0.3)
        
        class_label = {-1: "🔴 Destructive", 0: "🟡 Equilibrium", 1: "🟢 Constructive"}[predicted]
        
        fig.update_layout(
            title=(
                f"H7 Classification | ρ={rho:.3f}, v={v:.3f} → "
                f"{class_label} | Expected: {expected:+d} | {match}"
            ),
            template="plotly_dark",
            height=450,
            xaxis_title="Evolution Step",
            yaxis_title="Lagrangian / State",
            hovermode="x unified"
        )
        
        fig.show()
        
        # Print explanation
        print(f"\n📋 Ticket #{ticket['id']}: {ticket['description']}")
        print(f"Context: {ticket['context']}")
        print(f"\n🎯 Oracle Extraction:")
        print(f"   ρ = {rho:.4f} (severity)")
        print(f"   v = {v:+.4f} (intent)")
        print(f"\n📊 H7 Evolution:")
        print(f"   ψ₀ = {clf.history_psi[1]:+.4f}")
        print(f"   ψ(t) = {clf.history_psi[-1]:+.4f}")
        print(f"   Predicted: {class_label}")
        print(f"   Expected: {expected:+d}")
        print(f"   {match}")
        print(f"\n💡 {ticket['reasoning']}")


# Attach listeners
w_ticket.observe(update_widget_plot, names='value')
w_steps.observe(update_widget_plot, names='value')
w_dt.observe(update_widget_plot, names='value')
w_epsilon.observe(update_widget_plot, names='value')

print("✓ Update function defined and listeners attached")

✓ Update function defined and listeners attached


# Part 6: Interactive 
**Display everything**

In [8]:
# Create UI
header = widgets.HTML(
    "<h2 style='color: #7F77DD;'>⚛️ H7 Metriplectic QML Inspector</h2>"
    "<p style='color: #888;'>Zero-Shot Explainable Classification for Parking Violations</p>"
)

controls = widgets.VBox([
    widgets.HTML("<h3 style='color: #1D9E75;'>Controls</h3>"),
    widgets.HBox([widgets.Label("Ticket:"), w_ticket]),
    widgets.HBox([w_steps, w_dt, w_epsilon])
])

interface = widgets.VBox([header, controls, output_plot])

display(interface)

# Initial plot
update_widget_plot(None)

# Part 7: Batch Analysis (Optional)
**Evaluate all tickets**

In [9]:
print("\n" + "="*80)
print("BATCH ANALYSIS: All Tickets")
print("="*80 + "\n")

results = []

for ticket in sample_data:
    metrics = get_cached_metrics(ticket["description"])
    clf = H7TernaryClassifier(epsilon=0.25)
    predicted = clf.fit_predict(metrics["rho"], metrics["v"], steps=100, dt=-0.1)
    expected = ticket["expected_class"]
    match = "✅" if predicted == expected else "❌"
    
    results.append({
        "ID": ticket["id"],
        "Description": ticket["description"][:45] + "...",
        "ρ": f"{metrics['rho']:.3f}",
        "v": f"{metrics['v']:+.3f}",
        "Predicted": {-1: "🔴", 0: "🟡", 1: "🟢"}[predicted],
        "Expected": {-1: "🔴", 0: "🟡", 1: "🟢"}[expected],
        "Match": match
    })

df = pd.DataFrame(results)
display(df)

# Summary
print("\n" + "="*80)
matches = sum(1 for r in results if "✅" in r["Match"])
total = len(results)
accuracy = matches / total * 100

print(f"Total:     {total}")
print(f"Correct:   {matches}/{total}")
print(f"Accuracy:  {accuracy:.1f}%")
print("="*80)


BATCH ANALYSIS: All Tickets



,ID,Description,ρ,v,Predicted,Expected,Match
0,1,"Vehicle double-parked on fire hydrant, blocki...",-0.170,+0.472,🟡,🔴,❌
1,2,Expired parking meter by 3 minutes...,-0.631,+0.505,🟡,🟢,❌
2,3,Parked in no-parking zone during construction...,-0.122,+0.630,🟡,🟢,❌
3,4,Vehicle blocking wheelchair accessible ramp...,-0.240,+0.101,🟡,🔴,❌
4,5,Parked 18 inches from fire hydrant...,-0.998,+0.164,🟡,🟡,✅
5,6,Repeat offender: 7th citation in same locatio...,-0.501,+0.985,🟡,🔴,❌
6,7,Parked in restricted zone due to medical emer...,-0.849,+0.267,🟡,🔴,❌
7,8,Commercial vehicle in residential parking ove...,-0.601,+0.364,🟡,🟡,✅



Total:     8
Correct:   2/8
Accuracy:  25.0%


# Part 8: Export Results to CSV
### Generate submission-ready CSV file

In [10]:
# Create detailed submission CSV
print("\n" + "="*80)
print("EXPORTING RESULTS TO CSV")
print("="*80 + "\n")

submission_data = []

for ticket in sample_data:
    metrics = get_cached_metrics(ticket["description"])
    clf = H7TernaryClassifier(epsilon=0.25)
    predicted = clf.fit_predict(metrics["rho"], metrics["v"], steps=50, dt=0.1)
    expected = ticket["expected_class"]
    
    # Map class to label
    class_label = {-1: "Destructive", 0: "Equilibrium", 1: "Constructive"}[predicted]
    confidence = abs(clf.history_psi[1])  # Use final |ψ| as confidence
    
    submission_data.append({
        "ticket_id": ticket["id"],
        "description": ticket["description"],
        "context": ticket["context"],
        "oracle_rho": round(metrics["rho"], 4),
        "oracle_velocity": round(metrics["v"], 4),
        "initial_psi": round(clf.history_psi[0], 4),
        "final_psi": round(clf.history_psi[1], 4),
        "predicted_class": predicted,
        "predicted_label": class_label,
        "expected_class": expected,
        "confidence_score": round(confidence, 4),
        "is_correct": 1 if predicted == expected else 0,
        "reasoning": ticket["reasoning"]
    })

# Create DataFrame
submission_df = pd.DataFrame(submission_data)

# Display
print("Submission DataFrame:")
display(submission_df)

# Save to CSV
csv_filename = "h7_metriplectic_qml_predictions.csv"
submission_df.to_csv(csv_filename, index=False)

print(f"\n✓ CSV exported to: {csv_filename}")
print(f"✓ Rows: {len(submission_df)}")
print(f"✓ Columns: {list(submission_df.columns)}")

# Calculate statistics
total_correct = submission_df["is_correct"].sum()
total_tickets = len(submission_df)
accuracy_pct = (total_correct / total_tickets) * 100
avg_confidence = submission_df["confidence_score"].mean()

print(f"\n📊 Summary:")
print(f"   Total Predictions: {total_tickets}")
print(f"   Correct: {total_correct}/{total_tickets}")
print(f"   Accuracy: {accuracy_pct:.1f}%")
print(f"   Avg Confidence: {avg_confidence:.4f}")

# Class distribution
print(f"\n🎯 Predicted Class Distribution:")
class_dist = submission_df["predicted_label"].value_counts()
for cls, count in class_dist.items():
    pct = (count / total_tickets) * 100
    print(f"   {cls}: {count} ({pct:.1f}%)")

print(f"\n" + "="*80)
print(f"Ready for submission! Download: {csv_filename}")
print("="*80)


EXPORTING RESULTS TO CSV

Submission DataFrame:


,ticket_id,description,context,oracle_rho,oracle_velocity,initial_psi,final_psi,predicted_class,predicted_label,expected_class,confidence_score,is_correct,reasoning
0,1,"Vehicle double-parked on fire hydrant, blockin...","Rush hour, school zone, driver unavailable",-0.1702,0.4718,0,0,0,Equilibrium,-1,0,0,"Life safety risk, malicious impact"
1,2,Expired parking meter by 3 minutes,Driver returned immediately after noticing,-0.6313,0.5050,0,0,0,Equilibrium,1,0,0,Trivial administrative violation
2,3,Parked in no-parking zone during construction ...,"City closed normal street, signage unclear",-0.1218,0.6298,0,0,0,Equilibrium,1,0,0,Mitigating circumstances - infrastructure failure
3,4,Vehicle blocking wheelchair accessible ramp,Disabled resident unable to exit building,-0.2400,0.1007,0,0,0,Equilibrium,-1,0,0,Discriminatory impact on vulnerable population
4,5,Parked 18 inches from fire hydrant,"Hydrant obscured by snow, visibility issue",-0.9975,0.1637,0,0,0,Equilibrium,0,0,1,Technical violation with mitigating circumstances
5,6,Repeat offender: 7th citation in same location,Pattern indicates intentional non-compliance,-0.5008,0.9847,0,0,0,Equilibrium,-1,0,0,Willful pattern shows disregard for rules
6,7,Parked in restricted zone due to medical emerg...,Transporting injured family member to hospital,-0.8494,0.2667,0,0,0,Equilibrium,-1,0,0,Life-threatening emergency justifies violation
7,8,Commercial vehicle in residential parking over...,No permits obtained despite available process,-0.6011,0.3643,0,0,0,Equilibrium,0,0,1,Commercial activity without proper authorization



✓ CSV exported to: h7_metriplectic_qml_predictions.csv
✓ Rows: 8
✓ Columns: ['ticket_id', 'description', 'context', 'oracle_rho', 'oracle_velocity', 'initial_psi', 'final_psi', 'predicted_class', 'predicted_label', 'expected_class', 'confidence_score', 'is_correct', 'reasoning']

📊 Summary:
   Total Predictions: 8
   Correct: 2/8
   Accuracy: 25.0%
   Avg Confidence: 0.0000

🎯 Predicted Class Distribution:
   Equilibrium: 8 (100.0%)

Ready for submission! Download: h7_metriplectic_qml_predictions.csv
